In [7]:
import pandas as pd

ts_data = pd.read_parquet("../data/transformed/transformed_ts_2024_06.parquet")
ts_data

,pickup_hour,rides,pickup_location_id
0,2024-06-01 00:00:00,34,4
1,2024-06-01 01:00:00,63,4
2,2024-06-01 02:00:00,56,4
3,2024-06-01 03:00:00,30,4
4,2024-06-01 04:00:00,13,4
...,...,...,...
187195,2024-06-30 19:00:00,0,176
187196,2024-06-30 20:00:00,0,176
187197,2024-06-30 21:00:00,0,176
187198,2024-06-30 22:00:00,0,176


In [20]:
ts_data_one_location = ts_data.loc[ts_data.pickup_location_id == 43, :].reset_index(drop=True)
ts_data_one_location.head(25)

,pickup_hour,rides,pickup_location_id
0,2024-06-01 00:00:00,20,43
1,2024-06-01 01:00:00,0,43
2,2024-06-01 02:00:00,1,43
3,2024-06-01 03:00:00,7,43
4,2024-06-01 04:00:00,2,43
5,2024-06-01 05:00:00,1,43
6,2024-06-01 06:00:00,8,43
7,2024-06-01 07:00:00,21,43
8,2024-06-01 08:00:00,34,43
9,2024-06-01 09:00:00,53,43


In [9]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int,
    step_size: int
    ) -> list:

        stop_position = len(data) - 1
        
        # Start the first sub-sequence at index position 0
        subseq_first_idx = 0
        subseq_mid_idx = n_features
        subseq_last_idx = n_features + 1
        indices = []
        
        while subseq_last_idx <= stop_position:
            indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))
            
            subseq_first_idx += step_size
            subseq_mid_idx += step_size
            subseq_last_idx += step_size

        return indices

In [10]:
n_features = 24
step_size = 1

indices = get_cutoff_indices(ts_data_one_location, n_features, step_size)
indices[:5]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29)]

In [17]:
import numpy as np
print(np.__version__)

n_examples = len(indices)
x = np.ndarray(shape=(n_examples, n_features), dtype=np.float32)
y = np.ndarray(shape=(n_examples), dtype=np.float32)
pickup_hours = []

for i, idx in enumerate(indices):
    x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    y[i] = ts_data_one_location.iloc[idx[1]]['rides']  # <-- fixed: scalar access
    pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

2.4.3


In [18]:
print(f'{x.shape=}')
print(f'{x=}')
print(f'{pickup_hours[:5]=}')

x.shape=(695, 24)
x=array([[ 20.,   0.,   1., ...,  94.,  54.,  58.],
       [  0.,   1.,   7., ...,  54.,  58.,  22.],
       [  1.,   7.,   2., ...,  58.,  22.,  10.],
       ...,
       [118., 142.,  81., ..., 118.,  48.,  36.],
       [142.,  81.,  22., ...,  48.,  36.,  51.],
       [ 81.,  22.,  21., ...,  36.,  51.,  81.]],
      shape=(695, 24), dtype=float32)
pickup_hours[:5]=[Timestamp('2024-06-02 00:00:00'), Timestamp('2024-06-02 01:00:00'), Timestamp('2024-06-02 02:00:00'), Timestamp('2024-06-02 03:00:00'), Timestamp('2024-06-02 04:00:00')]


In [19]:
features_one_location = pd.DataFrame(
    x,
    columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(n_features))]
)
features_one_location

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_10_hour,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour
0,20.0,0.0,1.0,7.0,2.0,1.0,8.0,21.0,34.0,53.0,...,179.0,195.0,198.0,221.0,210.0,188.0,142.0,94.0,54.0,58.0
1,0.0,1.0,7.0,2.0,1.0,8.0,21.0,34.0,53.0,83.0,...,195.0,198.0,221.0,210.0,188.0,142.0,94.0,54.0,58.0,22.0
2,1.0,7.0,2.0,1.0,8.0,21.0,34.0,53.0,83.0,166.0,...,198.0,221.0,210.0,188.0,142.0,94.0,54.0,58.0,22.0,10.0
3,7.0,2.0,1.0,8.0,21.0,34.0,53.0,83.0,166.0,134.0,...,221.0,210.0,188.0,142.0,94.0,54.0,58.0,22.0,10.0,4.0
4,2.0,1.0,8.0,21.0,34.0,53.0,83.0,166.0,134.0,106.0,...,210.0,188.0,142.0,94.0,54.0,58.0,22.0,10.0,4.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690,174.0,138.0,118.0,142.0,81.0,22.0,21.0,6.0,6.0,1.0,...,19.0,47.0,32.0,87.0,119.0,157.0,158.0,209.0,229.0,118.0
691,138.0,118.0,142.0,81.0,22.0,21.0,6.0,6.0,1.0,3.0,...,47.0,32.0,87.0,119.0,157.0,158.0,209.0,229.0,118.0,48.0
692,118.0,142.0,81.0,22.0,21.0,6.0,6.0,1.0,3.0,4.0,...,32.0,87.0,119.0,157.0,158.0,209.0,229.0,118.0,48.0,36.0
693,142.0,81.0,22.0,21.0,6.0,6.0,1.0,3.0,4.0,3.0,...,87.0,119.0,157.0,158.0,209.0,229.0,118.0,48.0,36.0,51.0


In [21]:
targets_one_location = pd.DataFrame(y, columns=[f'target_rides_next_hour'])
targets_one_location

,target_rides_next_hour
0,22.0
1,10.0
2,4.0
3,2.0
4,0.0
...,...
690,48.0
691,36.0
692,51.0
693,81.0


In [26]:
from tqdm import tqdm

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'pickup_hour', 'rides', 'pickup_location_id'}

    location_ids = ts_data['pickup_location_id'].unique()
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for location_id in tqdm(location_ids):
        
        # keep only ts data for this `location_id`
        ts_data_one_location = ts_data.loc[
            ts_data.pickup_location_id == location_id, 
            ['pickup_hour', 'rides']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        pickup_hours = []
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i] = ts_data_one_location.iloc[idx[1]]['rides']
            # y[i] = ts_data_one_location.iloc[idx[1]:idx[2]]['rides'].values
            pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

        # numpy -> pandas
        features_one_location = pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_location['pickup_hour'] = pickup_hours
        features_one_location['pickup_location_id'] = location_id

        # numpy -> pandas
        targets_one_location = pd.DataFrame(y, columns=[f'target_rides_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_location])
        targets = pd.concat([targets, targets_one_location])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_rides_next_hour']

In [27]:
features, targets = transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len=24*7*1, # one week of history
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 260/260 [00:00<00:00, 284.03it/s]

features.shape=(5980, 170)
targets.shape=(5980,)


In [28]:
features.head()

,rides_previous_168_hour,rides_previous_167_hour,rides_previous_166_hour,rides_previous_165_hour,rides_previous_164_hour,rides_previous_163_hour,rides_previous_162_hour,rides_previous_161_hour,rides_previous_160_hour,rides_previous_159_hour,...,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id
0,34.0,63.0,56.0,30.0,13.0,0.0,2.0,5.0,2.0,7.0,...,12.0,11.0,12.0,10.0,11.0,7.0,13.0,36.0,2024-06-08,4
1,42.0,48.0,69.0,16.0,9.0,1.0,1.0,1.0,3.0,1.0,...,16.0,12.0,13.0,8.0,7.0,11.0,15.0,31.0,2024-06-09,4
2,0.0,0.0,2.0,0.0,0.0,1.0,6.0,11.0,18.0,7.0,...,11.0,11.0,4.0,6.0,4.0,1.0,4.0,1.0,2024-06-10,4
3,3.0,4.0,5.0,0.0,0.0,2.0,5.0,5.0,14.0,9.0,...,3.0,2.0,3.0,4.0,1.0,3.0,4.0,2.0,2024-06-11,4
4,3.0,3.0,0.0,0.0,0.0,2.0,8.0,14.0,14.0,13.0,...,3.0,1.0,7.0,2.0,5.0,6.0,4.0,1.0,2024-06-12,4


In [29]:
targets.head()

0    48.0
1    32.0
2     1.0
3     0.0
4     7.0
Name: target_rides_next_hour, dtype: float32